# Movies Project 
Camille Pagaduan, BAN6003-001

## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

Import the data.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Load the csv files into labeled dataframes. 

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')


Organize the dataframes.

In [3]:

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

## Business Context

A content acquisition team studying what movie characteristics are associated with high audience ratings. 

## Business Research Question
What factors are associated with higher user ratings?

This question is important because a variety of factors may be associated with higher audience ratings and would help a content acquisition team make informed decisions on which movies to acquire based on data-driven insights. 

## Examining Raw Tables and Row Meaning

Tables that will be used : Movies, Ratings, Credits, Keywords, Links

### Movies
Each row represents 1 movie and contains metadata on its identification, title, language, release date, runtime, budget, revenue, popularity, voting, genre, and other characteristics. 

vote_average represents average vote score from TMDB on a 0-10 scale.

In [23]:
# Movies Table
display(movies.head())

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995


## Ratings
Each row represents 1 movie and its identifier, rating, time, and date of the rating. 

rating represents an individiual user's rating of a movie on a scale of 0-5.

In [24]:
# Ratings Table
display(ratings.head())

,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02
3,1,1129,2.0,1260759185,2009-12-14 02:53:05
4,1,1172,4.0,1260759205,2009-12-14 02:53:25


## Credits
Each row represents 1 movie and includes its TMDB identifier, director, and top cast. 

In [25]:
# Credits Table
display(credits.head())

,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret
3,31357,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine
4,11862,Charles Shyer,Steve Martin|Diane Keaton|Martin Short


## Keywords
Each row represents 1 movie by its TMDB Identifier and includes its keywords list. 

In [26]:
# Keywords Table
display(keywords.head())

,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men
3,31357,based on novel|interracial relationship|single...
4,11862,baby|midlife crisis|confidence|aging|daughter|...


## Links
Each row represents 1 movie and its movie ID, IMDB ID, and TMDB ID. 

In [27]:
# Links Table
display(links.head())

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


## Initial Row and Column Counts

In [18]:
# Movies
display("Movies:")
display(movies.shape)
display(movies.columns)

'Movies:'

(9082, 21)

Index(['movieId', 'tmdbId', 'imdb_id', 'title', 'original_title',
       'original_language', 'release_date', 'runtime', 'budget', 'revenue',
       'popularity', 'vote_average', 'vote_count', 'genres',
       'production_companies', 'production_countries', 'adult', 'status',
       'genres_list', 'primary_genre', 'release_year'],
      dtype='str')

In [19]:
# Ratings
display("Ratings:")
display(ratings.shape)
display(ratings.columns)

'Ratings:'

(99810, 5)

Index(['userId', 'movieId', 'rating', 'timestamp', 'rating_datetime'], dtype='str')

In [20]:
# Credits
display("Credits:")
display(credits.shape)
display(credits.columns)

'Credits:'

(9082, 3)

Index(['tmdbId', 'director', 'top_cast'], dtype='str')

In [21]:
# Keywords
display("Keywords:")
display(keywords.shape)
display(keywords.columns)

'Keywords:'

(9082, 2)

Index(['tmdbId', 'keywords_list'], dtype='str')

In [22]:
# Links
display("Links:")
display(links.shape)
display(links.columns)

'Links:'

(9125, 3)

Index(['movieId', 'imdbId', 'tmdbId'], dtype='str')

## Notes on Rows and Columns
The Movies table has 9,082 rows and 21 columns. Columns in this table include movieId, tmdbId, title, original_title, orignal_language, release_date, runtime, budget, revenue, popularity, vote_average, vote_count, genres, production_companies, production_countries, adult, status, genres_list, primary_genre, relase_year.

The Ratings table has 99,810 rows and 5 columns. Columns include userId, movieId, rating, timestamp, and rating_datetime.

The Credits table has 9,082 rows and 3 columns. Colums include tmdbId, director, and top_cast.

The Keywords table has 9,082 rows and 2 columns. The columns include tmdbId and keywords_list.

The Links table has 9,125 rows and 3 columns. The columns include movieId, imdbId, and tmdbId. 

## Data Types and Missing Values

### Movies

In [30]:
# Movies Data Types
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               9082 non-null   int64  
 1   tmdbId                9082 non-null   int64  
 2   imdb_id               9082 non-null   str    
 3   title                 9082 non-null   str    
 4   original_title        9082 non-null   str    
 5   original_language     9082 non-null   str    
 6   release_date          9082 non-null   str    
 7   runtime               9082 non-null   float64
 8   budget                9082 non-null   int64  
 9   revenue               9082 non-null   float64
 10  popularity            9082 non-null   float64
 11  vote_average          9082 non-null   float64
 12  vote_count            9082 non-null   float64
 13  genres                9082 non-null   str    
 14  production_companies  9082 non-null   str    
 15  production_countries  9082 non-n

In [29]:
# Movies Missing Values
movies.isna().sum()

movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64

### Ratings

In [ ]:
# Ratings Data Types
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 99810 entries, 0 to 99809
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   userId           99810 non-null  int64         
 1   movieId          99810 non-null  int64         
 2   rating           99810 non-null  float64       
 3   timestamp        99810 non-null  int64         
 4   rating_datetime  99810 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 3.8 MB


In [33]:
# Ratings Missing Values
ratings.isna().sum()

userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64

### Credits

In [34]:
# Credits Data Types
credits.info()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   tmdbId    9082 non-null   int64
 1   director  9058 non-null   str  
 2   top_cast  8992 non-null   str  
dtypes: int64(1), str(2)
memory usage: 213.0 KB


In [35]:
# Credits Missing Values
credits.isna().sum()

tmdbId       0
director    24
top_cast    90
dtype: int64

### Keywords

In [36]:
# Keywords Data Types
keywords.info()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   tmdbId         9082 non-null   int64
 1   keywords_list  8321 non-null   str  
dtypes: int64(1), str(1)
memory usage: 142.0 KB


In [37]:
# Keywords Missing Values
keywords.isna().sum()

tmdbId             0
keywords_list    761
dtype: int64

### Links

In [38]:
# Links Data Types
links.info()

<class 'pandas.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9125 non-null   int64  
 1   imdbId   9125 non-null   int64  
 2   tmdbId   9112 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 214.0 KB


In [6]:
# Links Missing Values
links.isna().sum()

movieId     0
imdbId      0
tmdbId     13
dtype: int64

In [8]:
# Inspect Duplicate Values after Excluding Missing Values
links["tmdbId"].dropna().duplicated().sum()

np.int64(0)

## Notes on Data Types and Missing Values

Movies
- There are 2 missing values in the status column.
- There are 35 missing values in both the genres_list and primary_genre columns.
- All 3 columns with missing values have str data type. 

Ratings
- There are no missing values in this table.

Credits
- There are 24 missing values in the director column.
- There are 90 missing values in the top_cast column.
- Both columns with missing values have str data type.

Keywords
- There are 761 missing values in the keywords_list column.
- This column is a str data type.

Links
- There are 13 missing values in the tmdbId column. After excluding missing values, there were 0 duplicates. 
    The 13 missing values represent missing TMDB identifiers rather than duplicate values.  
- The tmdbId has a datatype of Float64 here, when it has int64 datatype in other tables.

## Duplicate Checks

### Duplicate Rows Check

In [42]:
# Check for Duplicate Rows
print("Movies Duplicate Rows:", movies.duplicated().sum())
print("Ratings Duplicate Rows:", ratings.duplicated().sum())
print("Credits Duplicate Rows:", credits.duplicated().sum())
print("Keywords Duplicate Rows:", keywords.duplicated().sum())
print("Links Duplicate Rows:", links.duplicated().sum())

Movies Duplicate Rows: 0
Ratings Duplicate Rows: 0
Credits Duplicate Rows: 0
Keywords Duplicate Rows: 0
Links Duplicate Rows: 0


### Notes on Duplicate Rows
There are no duplicate rows in any of the tables.

### Duplicate ID's Check

In [46]:
# Check for Duplicate ID's
print("Movie ID's Duplicate Rows:", movies['movieId'].duplicated().sum())
print("Movie TMDB ID's Duplicate Rows:", movies['tmdbId'].duplicated().sum())
print("Movie IMDB ID's Duplicate Rows:", movies['imdb_id'].duplicated().sum())

print("Rating User ID's Duplicate Rows:", ratings['userId'].duplicated().sum())
print("Rating Movie ID's Duplicate Rows:", ratings['movieId'].duplicated().sum())

print("Credit TMDB ID's Duplicate Rows:", credits['tmdbId'].duplicated().sum())

print("Keyword TMDB ID's Duplicate Rows:", keywords['tmdbId'].duplicated().sum())

print("Link Movie ID's Duplicate Rows:", links['movieId'].duplicated().sum())
print("Link IMDB ID's Duplicate Rows:", links['imdbId'].duplicated().sum())
print("Link TMDB ID's Duplicate Rows:", links['tmdbId'].duplicated().sum())


Movie ID's Duplicate Rows: 0
Movie TMDB ID's Duplicate Rows: 0
Movie IMDB ID's Duplicate Rows: 0
Rating User ID's Duplicate Rows: 99139
Rating Movie ID's Duplicate Rows: 90785
Credit TMDB ID's Duplicate Rows: 0
Keyword TMDB ID's Duplicate Rows: 0
Link Movie ID's Duplicate Rows: 0
Link IMDB ID's Duplicate Rows: 0
Link TMDB ID's Duplicate Rows: 12


In [47]:
# Check if same User has duplicate entries for the same Movie
print("Duplicate movie rating entries for same User ID:", ratings.duplicated(subset=['userId', 'movieId']).sum())

Duplicate movie rating entries for same User ID: 0


### Notes on Duplicate ID's
There are 99,129 duplicate user ID's and 90,785 duplicate movie ID's in the Rating table. It is normal to have duplicate user ID because the same user can give ratings to different movies and it will be counted as each separate rows/entries. The same movie can be rated by different users and can lead to duplicate movie ID's. 

There are 12 duplicate ID's in the Link table for TMDB arise from the missing identifiers and not duplicate values as proven when inspecting the Links table in the previous section. 

## Checking for Suspicious Values

### Checking the Min and Max Values

In [20]:
## Checking for Suspicious Values 

## Check what is the min and max values
# Min and Max Vote Average
print("Minimum vote average:", movies['vote_average'].min())
print("Maximum vote average:", movies['vote_average'].max())

# Min and Max Vote Count
print("\nMinimum vote count:", movies['vote_count'].min())
print("Maximum vote count:", movies['vote_count'].max())

# Min and Max Budget
print("\nMiniumum budget:", movies['budget'].min())
print("Maxiumum budget:", movies['budget'].max())

# Min and Max Revenue
print("\nMiniumum revenue:", movies['revenue'].min())
print("Maxiumum revenue:", movies['revenue'].max())

# Min and Max Release Year
print("\nMiniumum release year:", movies['release_year'].min())
print("Maxiumum release year:", movies['release_year'].max())

Minimum vote average: 0.0
Maximum vote average: 10.0

Minimum vote count: 0.0
Maximum vote count: 14075.0

Miniumum budget: 0
Maxiumum budget: 380000000

Miniumum revenue: 0.0
Maxiumum revenue: 2787965087.0

Miniumum release year: 1902
Maxiumum release year: 2016


### Notes on Min and Max Values
- All vote averages fall within the 0 to 10 range. 
- The vote count ranges from 0 to 14,075.
- The budget ranges from 0 to 380,000,000.
- The revenue ranges from 0 to 2,787,965,087.
- The minimum release year is 1902 and maximum release year is 2016.


### Checking the Rating Range

In [9]:
# Check the Rating Range in Movies Table
print("Movie Table Ratings below 0:", (movies['vote_average'] < 0).sum())
print("Movie Table Ratings above 10:", (movies['vote_average'] > 10).sum())

print("\nRatings Table Ratings below 0:", (ratings['rating'] < 0).sum())
print("Ratings Table Ratings above 5:", (ratings['rating'] > 5).sum())

Movie Table Ratings below 0: 0
Movie Table Ratings above 10: 0

Ratings Table Ratings below 0: 0
Ratings Table Ratings above 5: 0


### Notes on Rating Range
- The Movie table rating values fall within the 0-10 range. 
- The Ratings table rating values fall within the 0-5 range.
- There are no outlier ratings for either table. 

### Checking the Runtime

In [21]:
# Check the Runtime
print("Negative runtimes:", (movies['runtime'] < 0).sum())

# Check for High Runtimes
print("Runtimes over 300 minutes:", (movies['runtime'] > 300).sum())

# Check Min and Max Runtimes
print("\nMinimum runtimes:", (movies['runtime'].min()))
print("Maximum runtimes:", (movies['runtime'].max()))

Negative runtimes: 0
Runtimes over 300 minutes: 15

Minimum runtimes: 0.0
Maximum runtimes: 1140.0


### Notes on Runtime
- There are no negative runtimes.
- There are 15 movies with runtimes over 300 minutes.
- The minimum runtime is 0 and the maximum runtime is 1,140 minutes, which should be investigated further. 

### Checking for Value Counts

In [15]:
# Check for status value counts
print("Status counts:")
print(movies['status'].value_counts())

# Check for original language value counts
print("\nOriginal language values:")
print(movies['original_language'].value_counts())

# Check for adult value counts
print("\nAdult values:")
print(movies['adult'].value_counts())

# Check for primary genre value counts
print("\nPrimary genre values:")
print(movies['primary_genre'].value_counts())

Status counts:
status
Released           9061
Rumored              11
Post Production       7
In Production         1
Name: count, dtype: int64

Original language values:
original_language
en    7947
fr     272
ja     185
de     113
it      98
es      88
cn      47
sv      41
zh      39
ko      31
da      30
ru      27
pt      25
nl      16
fa      14
fi      13
hi      13
cs      10
tr       8
no       7
th       7
he       6
pl       6
el       5
sr       4
hu       4
bn       3
xx       3
id       3
vi       2
nb       2
ro       2
ar       2
bo       1
sk       1
bs       1
is       1
ps       1
uk       1
af       1
et       1
lo       1
Name: count, dtype: int64

Adult values:
adult
False    9082
Name: count, dtype: int64

Primary genre values:
primary_genre
Drama              2413
Comedy             2168
Action             1177
Adventure           517
Horror              497
Crime               417
Documentary         383
Thriller            263
Fantasy             244
Animation

### Notes on Value Counts
- In the Status column there are 9,061 released movies, 11 rumored movies, 7 in post production movies, and 1 in production movie.
- In the original language column, en has 7,947 values.
- In the primary genre column, there 44 labeled as History, 18 as TV Movie, and 1 as Foreign. 

### Validating IMDB ID and TMDB ID

In [10]:
#Comparing IMDB ID from Movies and Links Table

print("Movies table:")
print(movies["imdb_id"].head())

print("\nLinks table:")
print(links["imdbId"].head())

Movies table:
0    tt0114709
1    tt0113497
2    tt0113228
3    tt0114885
4    tt0113041
Name: imdb_id, dtype: str

Links table:
0    114709
1    113497
2    113228
3    114885
4    113041
Name: imdbId, dtype: int64


## Early Data Quality Issues
Column Names
- The Movies table column is labeled imdb_id, while the Links table column is labled imdbId. These columns appear to represent the same identifier (IMDB ID), but their naming conventions differ. The correct column name must be referenced when joining or comparing tables. 

Rating Measures
- The Movies table has a column called vote_average, which represents the average TMDB vote score from a 0-10 rating range.
- The Ratings table has a column called rating, which represents individual MovieLens user ratings from a 0-5 rating range. 
- These variables are derived from different platforms and are not interchangeable. 
- Only one rating source will be selected as the dependent value. 

Data Types
- The tmdbId column in the Links table has a float64 datatype, while tmdbId columns in other tables has int64 datatype. This difference will reviewed to determine if the columns need to be standardized before joining or comparing tables.

Missing Values
- The Movies table has 2 missing values in the status column and 35 missing values in both the genre and primary_genre columns. These missing values need to be evaluated to determine how they may affect analysis and how they should be handled during data cleaning.
- The Credits table has 24 missing values in director and 90 missing values in top_cast. These missing values need to be evaluated to determine their importance to the research question and how they should be handled.
- The Keywords table has 761 missing values in keywords_list. These missing values need to be evaluated to determine how they should be handled before using this table in the analysis. 
- The Links table has 13 missing tmdbId values. These records are missing a TMDB identifier, which may affect the ability to effectively match them with the Movies table during joining.

Duplicate Rows
- The initial check showed 12 tmdbId values that could be duplicated or missing in the Links table. After inspecting with dropna(), no duplicated values were identified after excluding missing values. There are 13 missing identifiers that should be treated as missing values rather than duplicate records.

Suspicious Values
- The minimum value for vote_count, vote average, budget, revenue, and runtime is 0. These values need to be evaluated to determine whether 0 represents a valid value or missing/unknown information for each variable.
- The maximum runtime value is 1,140 minutes which is unusually high value.This value should be investigated to determine whether it represents a legtimate movie runtime or potential data-quality issue. 

## Inital Cleaning Plan
- Standardize ID formatting : imdb_id and imdbId by accounting for tt prefix
- Standardize data types : tmdbId in the links table

- Handle missing values : status, genres, primary_genre, director, top_cast, keywords_list, tmdbId
- Evaluate these missing values by retaining, removing, or handling missing values accordingly and based on number of affected records

- Review duplicate identifiers : tmdbId in links table
- Remove records only after determining appropriate row-level meaning and if represent legitimate or duplicate records.

- Investigate suspicious numerical values
- Review 0 values in vote_count, vote_average, budget, revenue, and runtime.
- Determine if valid values or missing/unknown information. 
- Investigate high value of 1,140 minutes runtime.

- Review rating measures : TMDB and MovieLens are different rating sources and scales
-Select one rating source a dependent variable for analysis

- Preserve raw data by creating and working on a copy version when performing cleaning and transformations.
- Document cleaning decisions made. 

## Proposed Target / Outcome
- The proposed target is audience rating, which is represented by the vote_average column in the movies table.
- The analysis will focus on identifying what movie characteristics are associated with higher audience ratings. 

## Early Ethics and Governance Concerns
User Privacy
- The ratings data has user ID, which will be kept pseudonymous. Individual users won't be identifiable and analysis will focus on aggregate movie-level ratings.

Representativness
- User ratings may not represent the ratings of all movie viewers. This only includes users who have submitted a rating. Results are interpreted as patterns within this population rather than it being representative of all movie audience.

Data Quality and Transparency
- Values that are missing, duplicated, and suspicious have been identified. Cleaning decisions will be done accordingly and documented to reduce risk of unintended bias.

Data Integration
- Movie information will come from multiple different tables that use different data types and id. Identifier mappings will be verified before joining any tables to reduce incorrect use of the information. 

## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



## Data Cleaning 

### Create a Working Copy

In [4]:
# Create working copies of raw datasets
movies_clean = movies.copy()
ratings_clean = ratings.copy()
links_clean = links.copy()
credits_clean = credits.copy()
keywords_clean = keywords.copy()

### Standardization 

In [16]:
# Standardize formatting of Imdb ID in Movies table 
movies_clean["imdb_id_clean"] = (
    movies_clean["imdb_id"]
    .str.replace("tt", "", regex=False)
    .str.lstrip("0")
)

movies_clean[["imdb_id", "imdb_id_clean"]].head(10)

,imdb_id,imdb_id_clean
0,tt0114709,114709
1,tt0113497,113497
2,tt0113228,113228
3,tt0114885,114885
4,tt0113041,113041
5,tt0113277,113277
6,tt0114319,114319
7,tt0112302,112302
8,tt0114576,114576
9,tt0113189,113189


In [12]:
# Convert Movies ID to match datatype in Links ID
movies_clean["imdb_id_clean"] = (
    movies_clean["imdb_id_clean"]
    .astype("Int64")
)

In [13]:
# Check cleaned IDs and data types
print("movies imdb_id:", movies_clean["imdb_id_clean"].head(10))
print("\nlinks imdbId:", links_clean["imdbId"].head(10))

movies imdb_id: 0    114709
1    113497
2    113228
3    114885
4    113041
5    113277
6    114319
7    112302
8    114576
9    113189
Name: imdb_id_clean, dtype: Int64

links imdbId: 0    114709
1    113497
2    113228
3    114885
4    113041
5    113277
6    114319
7    112302
8    114576
9    113189
Name: imdbId, dtype: int64


In [15]:
# Rename Links imdbId to match Movies imdb_id 
links_clean = links_clean.rename(columns={
    "imdbId" : "imdb_id"
})

links_clean[["imdb_id"]].head(10)

,imdb_id
0,114709
1,113497
2,113228
3,114885
4,113041
5,113277
6,114319
7,112302
8,114576
9,113189


### Notes on Standardization
- Standardized imdb_id in Movies table to match format of imdbId in Links table
- Standardized imdb_id format by removing "tt" and leading "0" prefixes
- Converted imdb_id data type to Int64 to match data type of imdbId in Links table
- Renamed imdbId column in Links table to match imdb_id column in Movies table

## Addressing Missing Values

In [19]:
# View missing values in Movies Table
movies_clean.loc[
    movies_clean[["status", "genres", "primary_genre"]].isna().any(axis=1)
].head(10)


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,imdb_id_clean
78,84,188588,tt0113612,Last Summer in the Hamptons,Last Summer in the Hamptons,en,1995-11-22,108.0,0,0.0,0.531159,0.0,0.0,[],[],[],False,Released,NaN,NaN,1995,113612
648,787,52059,tt0113147,The Gate of Heavenly Peace,The Gate of Heavenly Peace,en,1995-01-01,180.0,0,0.0,0.107050,6.7,3.0,"[{'id': 99, 'name': 'Documentary'}]",[],[],False,NaN,Documentary,Documentary,1995,113147
650,793,124306,tt0106810,My Life and Times With Antonin Artaud,En compagnie d'Antonin Artaud,en,1993-01-01,90.0,0,0.0,0.000308,8.0,1.0,[],[],[],False,Released,NaN,NaN,1993,106810
683,844,297645,tt0108227,The Story of Xinghua,The Story of Xinghua,en,1994-03-30,90.0,0,0.0,0.001652,0.0,0.0,[],[],[],False,Released,NaN,NaN,1994,108227
684,845,48144,tt0111424,The Day the Sun Turned Cold,天國逆子,zh,1994-01-01,99.0,0,0.0,0.076062,7.0,2.0,[],[],"[{'iso_3166_1': 'HK', 'name': 'Hong Kong'}]",False,Released,NaN,NaN,1994,111424
695,865,42758,tt0114474,Small Faces,Small Faces,en,1996-04-05,108.0,1850000,0.0,0.909130,6.4,10.0,[],[],"[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}]",False,Released,NaN,NaN,1996,114474
710,889,101230,tt0109001,1-900,06,nl,1994-09-01,0.0,0,0.0,0.062128,5.0,1.0,[],[],[],False,Released,NaN,NaN,1994,109001
875,1087,21202,tt0113731,Madame Butterfly,Madame Butterfly,en,1995-11-22,135.0,0,0.0,0.148900,4.8,3.0,[],[],[],False,Released,NaN,NaN,1995,113731
1229,1543,125021,tt0118964,The Designated Mourner,The Designated Mourner,en,1997-05-23,94.0,0,0.0,0.152272,8.0,2.0,[],[],[],False,Released,NaN,NaN,1997,118964
1464,1901,416437,tt0150290,Dear Jesse,Dear Jesse,en,1998-10-10,82.0,0,0.0,0.001354,0.0,0.0,[],[],[],False,Released,NaN,NaN,1998,150290


In [20]:
# View missing values in Credits Table
credits_clean.loc[
    credits_clean[["director", "top_cast"]].isna().any(axis=1)
].head(10)

,tmdbId,director,top_cast
216,43475,Art Clokey,NaN
392,24257,Bruce Brown,NaN
630,85778,Freida Lee Mock,NaN
648,52059,Carma Hinton,NaN
910,79306,NaN,Warren Miller
922,124676,Nina Menkes,NaN
1240,47686,NaN,David Arquette|Brad Hunt|Kathryn Erbe
1413,17644,Steve Gomer,NaN
1464,416437,Tim Kirkman,NaN
1840,18603,S.R. Bindler,NaN


In [21]:
# View missing values in Keywords Table
keywords_clean.loc[
    keywords_clean[["keywords_list"]].isna().any(axis=1)
].head(10)

,tmdbId,keywords_list
7,45325,NaN
35,139405,NaN
55,99040,NaN
71,124626,NaN
78,188588,NaN
90,45549,NaN
94,47475,NaN
99,89333,NaN
103,43566,NaN
109,32119,NaN


In [22]:
# View missing values in Links Table
links_clean.loc[
    links_clean[["tmdbId"]].isna().any(axis=1)
].head(10)

,movieId,imdb_id,tmdbId
607,720,118114,NaN
608,721,114103,NaN
640,769,116992,NaN
910,1133,111357,NaN
2282,2851,81454,NaN
3246,4051,56600,NaN
5826,26587,92337,NaN
7408,72781,1104746,NaN
7438,73759,495212,NaN
7588,79299,874957,NaN


### Flagging Missing Values

In [28]:
# Flag missing values in Movies Table
movies_clean["missing_status"] = movies_clean["status"].isna()
movies_clean["missing_genres"] = movies_clean["genres"].isna()
movies_clean["missing_primary_genre"] = movies_clean["primary_genre"].isna()

In [29]:
# Flag missing values in Credits Table
credits_clean["missing_director"] = credits_clean["director"].isna()
credits_clean["missing_top_cast"] = credits_clean["top_cast"].isna()

In [30]:
# Flag missing values in Keywords Table
keywords_clean["missing_keywords"] = keywords_clean["keywords_list"].isna()

In [31]:
# Flag missing values in Links Table
links_clean["missing_tmdbId"] = links_clean["tmdbId"].isna()

In [32]:
# Total Flagged Missing Values from each Table
print("Movies Flagged Values:")
print(movies_clean.filter(like="missing_").sum())

print("\nCredits Flagged Values:")
print(credits_clean.filter(like="missing_").sum())

print("\nKeywords Flagged Values:")
print(keywords_clean.filter(like="missing_").sum())

print("\nLinks Flagged Values:")
print(links_clean.filter(like="missing_").sum())

Movies Flagged Values:
missing_status            2
missing_genres            0
missing_primary_genre    35
dtype: int64

Credits Flagged Values:
missing_director    24
missing_top_cast    90
dtype: int64

Keywords Flagged Values:
missing_keywords    761
dtype: int64

Links Flagged Values:
missing_tmdbId    13
dtype: int64


## Notes on Addressing and Flagging Missing Values 
- Evaluated missing values from the Movies, Credits, Keywords, and Links tables.
- Final decision is to retain and flag the missing values.
- The total of flagged values consisted of 37 from Movies, 114 from Credits, 761 from Keywords, and 13 from Links tables.
- During analysis, if the missing variable is required, the individual record will be excluded.
- imdb_id will be used for matching records, meanig retaining missing tmdbId doesn't affect analysis.

## Addressing Zero Values

In [35]:
# View Zero Values in Movies Table
movies_clean.loc[
    (movies_clean[
        ["vote_count", "vote_average", "budget", "revenue", "runtime"]
        ]== 0).any(axis=1)
].head(5)

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,imdb_id_clean,missing_status,missing_genres,missing_primary_genre
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,113228,False,False,False
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,113041,False,False,False
6,7,11860,tt0114319,Sabrina,Sabrina,en,1995-12-15,127.0,58000000,0.0,6.677277,6.2,141.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...",False,Released,Comedy|Romance,Comedy,1995,114319,False,False,False
7,8,45325,tt0112302,Tom and Huck,Tom and Huck,en,1995-12-22,97.0,0,0.0,2.561161,5.4,45.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Drama|Family,Action,1995,112302,False,False,False
11,12,12110,tt0112896,Dracula: Dead and Loving It,Dracula: Dead and Loving It,en,1995-12-22,88.0,0,0.0,5.430331,5.7,210.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'Columbia Pictures', 'id': 5}, {'nam...","[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...",False,Released,Comedy|Horror,Comedy,1995,112896,False,False,False


In [40]:
# Create Flags for zero values in Movies Table
movies_clean["zero_vote_count"] = movies_clean["vote_count"] == 0
movies_clean["zero_vote_average"] = movies_clean["vote_average"] == 0
movies_clean["zero_budget"] = movies_clean["budget"] == 0
movies_clean["zero_revenue"] = movies_clean["revenue"] == 0
movies_clean["zero_runtime"] = movies_clean["runtime"] == 0

In [39]:
# Summary of total zero value flags in Movies Table
movies_zero_summary = movies_clean.filter(like="zero_").sum()
print(movies_zero_summary) 

zero_vote_count        44
zero_vote_average      48
zero_budget          4443
zero_revenue         4322
zero_runtime           20
dtype: int64


## Notes on Addressing Zero Values in Movies Table
- Evaluated zero values found in vote_count, vote_average, budget, revenue, and runtime variables in the Movies table.
- Zero values were flagged and will be retained to be investigated further
- During analysis, if variable with zero value is required, those records will be excluded. 

## Investigate Suspicious Values

In [47]:
# Inspect high runtimes in Movies Table
movies_clean.loc[
    movies_clean["runtime"] > 300,
    ["tmdbId", "title", "runtime", "release_date", "status", "genres", "primary_genre"]
].sort_values(by="runtime", ascending=False)

,tmdbId,title,runtime,release_date,status,genres,primary_genre
8181,142051,Jazz,1140.0,2001-01-09,Released,"[{'id': 99, 'name': 'Documentary'}]",Documentary
7224,45560,Berlin Alexanderplatz,931.0,1980-08-28,Released,"[{'id': 18, 'name': 'Drama'}]",Drama
5870,30983,From the Earth to the Moon,720.0,1998-04-05,Released,"[{'id': 99, 'name': 'Documentary'}, {'id': 18,...",Documentary
7981,26397,The Civil War,680.0,1990-09-23,Released,"[{'id': 99, 'name': 'Documentary'}]",Documentary
4862,42044,Shoah,566.0,1985-11-01,Released,"[{'id': 99, 'name': 'Documentary'}]",Documentary
8763,189197,The Pacific,540.0,2010-03-15,Released,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action
8477,54102,Generation Kill,470.0,2008-07-13,Released,"[{'id': 18, 'name': 'Drama'}, {'id': 36, 'name...",Drama
5881,110147,The 10th Kingdom,417.0,2000-02-25,Released,"[{'id': 12, 'name': 'Adventure'}, {'id': 35, '...",Adventure
8247,134481,Frozen Planet,392.0,2011-10-26,Released,"[{'id': 99, 'name': 'Documentary'}]",Documentary
5216,11659,The Best of Youth,366.0,2003-05-19,Released,"[{'id': 18, 'name': 'Drama'}]",Drama


In [48]:
# Flag records that are Miniseries or Documentary Series
movies_clean["non_feature_films"] = False

movies_clean.loc[
    movies_clean["tmdbId"].isin([
        142051, #Jazz
        45560, # Berlin Alexanderplatz
        30983, # From the Earth to the Moon
        26397, # The Civil War
        189197, # The Pacific
        54102, # Generation Kill
        110147, # The 10th Kingdom
        134481, # Frozen Planet
        11659, # The Best of Youth
        89049, # The Staircase
        24019, # Angels in America
        202241, # Mildred Pierce
        140207 # Tinker Tailor Soldier Spy
    ]),
    "non_feature_films"
] = True 

In [ ]:
# Verify Number of Flagged non_feature_films
movies_clean["non_feature_films"].value_counts()

non_feature_films
False    9069
True       13
Name: count, dtype: int64

In [50]:
# Create New DataFrame Copy that excludes non_feature_films
movies_analysis = movies_clean.loc[
    movies_clean["non_feature_films"] == False
].copy()

print("Cleaned dataset:", len(movies_clean))
print("Analysis dataset:", len(movies_analysis))

Cleaned dataset: 9082
Analysis dataset: 9069


## Notes on Suspicious Runtime Values
- 15 records showed runtime > 300 minutes. 
- 13 of the 15 movies are tv miniseries or documentary series.
- The 13 records were flagged and retained as non-feature films.
- Napolean (1927) and Shoah (1985) are considered legitimate long runtime feature films. 
- New dataframe, called movies_analysis, was created to exclude non-feature films.

## Creating Derived Variables

In [ ]:
# Create Release Year from Release Date
movies_analysis["release_year"] = (
    pd.to_datetime(movies_analysis["release_date"]).dt.year
)

In [ ]:
# Check Created Release Year Column
movies_analysis[["title", "release_date", "release_year"]].head(5)

,title,release_date,release_year
0,Toy Story,1995-10-30,1995
1,Jumanji,1995-12-15,1995
2,Grumpier Old Men,1995-12-22,1995
3,Waiting to Exhale,1995-12-22,1995
4,Father of the Bride Part II,1995-02-10,1995


## Notes on Derived Variables
- Created release_year variable from release_date.
- Release year will help target analysis based on different years movies were released. 

## Updated Unit of Analysis
One row is one movie. 

## Updated Business Question
What factors are associated with higher user ratings? (Based on the MovieLens Individual User Scores)

## Aggregation

In [56]:
# Aggregate ratings from MovieLens using movieId
movielens_rating = (
    ratings.groupby("movieId")
    .agg(
        movielens_rating_count=("rating", "count"),
        movielens_avg_rating=("rating", "mean")
    )
    .reset_index()
)

movielens_rating.head()

,movieId,movielens_rating_count,movielens_avg_rating
0,1,247,3.872470
1,2,107,3.401869
2,3,59,3.161017
3,4,13,2.384615
4,5,56,3.267857


In [21]:
# SQL Query: Aggregating MovieLens Ratings using movieId
query = """
SELECT 
    "movieId",
    COUNT(rating) AS movielens_rating_count,
    AVG(rating) AS movielens_average_count
FROM
    ratings
GROUP BY
    "movieId"
ORDER BY
    "movieId";
    """

movielens_rating = pd.read_sql(query, project_engine)

movielens_rating.head()

,movieId,movielens_rating_count,movielens_average_count
0,1,247,3.872470
1,2,107,3.401869
2,3,59,3.161017
3,4,13,2.384615
4,5,56,3.267857


In [57]:
# Validation Uniquness
print("Original ratings table:")
print("Unique movieId:", ratings["movieId"].is_unique)
print("Duplicate movieId:", ratings["movieId"].duplicated().sum()) 

print("\nAggregated MovieLens ratings table:")
print("Unique movieId:", movielens_rating["movieId"].is_unique)
print("Duplicate movieId:", movielens_rating["movieId"].duplicated().sum()) 

Original ratings table:
Unique movieId: False
Duplicate movieId: 90785

Aggregated MovieLens ratings table:
Unique movieId: True
Duplicate movieId: 0


## Notes on Aggregation
- Created 2 aggregations on the Movielens rating scores.
- Aggregated rating count represents the total number of user ratings for 1 movie.
- Aggregated average rating represents the average user rating a singular movie received. 
- Validation shows that there are only unique movieIds and no duplicate identifiers after aggregation.

## Merging Tables

In [61]:
print(links.columns.tolist())

['movieId', 'imdbId', 'tmdbId']


In [ ]:
# Validation before merging

# Check Row Counts
print("Row Counts:")
print("Credits:", len(credits_clean))
print("Keywords:", len(keywords_clean))
print("Links:", len(links_clean))

# Check Unique IDs
print("Unique Credits IDs:", credits_clean["tmdbId"].is_unique)
print("\nUnique Keywords IDs:", keywords_clean["tmdbId"].is_unique)

print("\nLinks Table:")
for col in ["movieId", "imdbId", "tmdbId"]:
    print(f"\n{col}:")
    print("Unique IDs:", links[col].is_unique)
    print("Duplicate IDs:", links[col].duplicated().sum())

Unique Credits IDs: True

Unique Keywords IDs: True

Links Table:

movieId:
Unique IDs: True
Duplicate IDs: 0

imdbId:
Unique IDs: True
Duplicate IDs: 0

tmdbId:
Unique IDs: False
Duplicate IDs: 12


## Create Movie-Level ABT

In [64]:
# Create movie-level ABT
movie_abt = movies_analysis.copy()

movie_abt.head()

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,imdb_id_clean,missing_status,missing_genres,missing_primary_genre,zero_vote_count,zero_vote_average,zero_budget,zero_revenue,zero_runtime,non_feature_films
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,114709,False,False,False,False,False,False,False,False,False
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,113497,False,False,False,False,False,False,False,False,False
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,113228,False,False,False,False,False,True,True,False,False
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,114885,False,False,False,False,False,False,False,False,False
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,113041,False,False,False,False,False,True,False,False,False


In [67]:
# Check ABT dataframe shape and uniqueness
print("Rows:", len(movie_abt))
print("Columns:", len(movie_abt.columns))
print("Unique Movie IDs:", movie_abt["movieId"].is_unique)

print("All Columns:", movie_abt.columns.tolist())

Rows: 9069
Columns: 31
Unique Movie IDs: True
All Columns: ['movieId', 'tmdbId', 'imdb_id', 'title', 'original_title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'genres', 'production_companies', 'production_countries', 'adult', 'status', 'genres_list', 'primary_genre', 'release_year', 'imdb_id_clean', 'missing_status', 'missing_genres', 'missing_primary_genre', 'zero_vote_count', 'zero_vote_average', 'zero_budget', 'zero_revenue', 'zero_runtime', 'non_feature_films']


In [68]:
movie_abt.info()

<class 'pandas.DataFrame'>
Index: 9069 entries, 0 to 9081
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   movieId                9069 non-null   int64  
 1   tmdbId                 9069 non-null   int64  
 2   imdb_id                9069 non-null   str    
 3   title                  9069 non-null   str    
 4   original_title         9069 non-null   str    
 5   original_language      9069 non-null   str    
 6   release_date           9069 non-null   str    
 7   runtime                9069 non-null   float64
 8   budget                 9069 non-null   int64  
 9   revenue                9069 non-null   float64
 10  popularity             9069 non-null   float64
 11  vote_average           9069 non-null   float64
 12  vote_count             9069 non-null   float64
 13  genres                 9069 non-null   str    
 14  production_companies   9069 non-null   str    
 15  production_countries

In [ ]:
# Merge MovieLens rating table into movie ABT
movie_abt= movie_abt.merge(
    movielens_rating,
    on="movieId",
    how="left",
    validate="one_to_one"
)

## Connecting SQL

In [12]:
# COPY - Path A, cell 1 of 3: connect to your own Neon database
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Add an empty PROJECT_DATABASE_URL line to .env the first time. Git ignores .env.
env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

# Saved query results go here, so the notebook also runs without your .env (see Section 3.4).
SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")

Connected to your own database. For safety, we do not print the connection string.


In [9]:
repo_root = REPO_ROOT

print(repo_root)

/workspaces/ban6003-camille-pagaduan-movies-project


In [13]:
# COPY - Path A, cell 2 of 3: upload a CSV file or a DataFrame to your database
def upload_table(source, table_name):
    """Upload a CSV file path or a DataFrame to your own database and check the row count."""
    if project_engine is None:
        print(f"Skipped uploading {table_name}: no PROJECT_DATABASE_URL in .env.")
        return
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, low_memory=False)
    df.to_sql(table_name, project_engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    status = "OK" if db_rows == len(df) else "CHECK: the row counts differ"
    print(f"{table_name}: {len(df):,} rows in Python, {db_rows:,} rows in the database. {status}")


# Fill in your own files. One line per table: upload_table(path_or_dataframe, "table_name")
# In your project, use your data files, for example:
upload_table(repo_root / "data" / "ratings.csv", "ratings")
upload_table(repo_root / "data" / "movies_metadata.csv", "movies")
upload_table(repo_root / "data" / "links.csv", "links")
upload_table(repo_root / "data" / "keywords_curated.csv", "keywords")
upload_table(repo_root / "data" / "credits_curated.csv", "credits")

upload_table(movie_abt, "movie_abt")      # a DataFrame you built in Pandas

ratings: 99,810 rows in Python, 99,810 rows in the database. OK
movies: 9,082 rows in Python, 9,082 rows in the database. OK
links: 9,125 rows in Python, 9,125 rows in the database. OK
keywords: 9,082 rows in Python, 9,082 rows in the database. OK
credits: 9,082 rows in Python, 9,082 rows in the database. OK


NameError: name 'movie_abt' is not defined

In [15]:
# COPY - Path A, cell 3 of 3: run SQL on your database and save the result
def run_project_sql(query, result_name):
    """Run SQL on your own database and save the result. Without a connection, show the saved result."""
    result_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        result = pd.read_sql(text(query), project_engine)
        result.to_csv(result_path, index=False)
        print(f"Ran on your database. Saved the result to {result_path.relative_to(repo_root)}")
        return result
    if result_path.exists():
        print(f"No database connection here, so this is the saved result from {result_path.relative_to(repo_root)}")
        return pd.read_csv(result_path)
    print(f"Skipped {result_name}: no PROJECT_DATABASE_URL and no saved result yet.")
    return pd.DataFrame()

In [6]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [7]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


Preliminary ABT shape: (9082, 32)


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,high_user_rating,well_rated_enough_data
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,1,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,0,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,0,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,0,0
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,0,0


,missing_rate
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
first_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
primary_genre,0.003854
genres_list,0.003854
director,0.002643


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
